# Lightweight TB-Net — MLRC finishing runs

Three things this notebook produces:

1. **Sensitivity-gap investigation** — split class-balance vs. paper Table 1, multi-seed eval of baseline
2. **75% pruning** — adds the third point to the sparsity curve the roadmap calls for
3. **Phone-capture robustness** — re-eval every model under CheXphoto-style augmentations (glare, blur, moiré, rotation, brightness)

Outputs: `results_clean.csv`, `results_phone.csv`, `figures/sparsity_tradeoff.png` (3 points), `figures/phone_robustness.png`.

Works on Kaggle (GPU T4) and locally — paths auto-detected.

## Cell 1 — Setup

In [ ]:
import os, sys, time, copy, glob, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v3_small
from PIL import Image, ImageFilter
from sklearn.metrics import confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt

KAGGLE = os.path.exists('/kaggle/input')
print('Kaggle:', KAGGLE)

# Resolves both the new (src/model.py + data_splits/) layout and the legacy
# (tbnet_pytorch.py + *_split_new.csv at root) layout that existing Kaggle datasets use.
def _find_model_module(root):
    for name in ('model.py', 'tbnet_pytorch.py'):
        for sub in ('', 'src', 'tbnet-kaggle', 'tbnet-kaggle/src'):
            p = os.path.join(root, sub) if sub else root
            if os.path.exists(os.path.join(p, name)):
                return p, name[:-3]
    return None, None

if KAGGLE:
    PREFERRED_DATA_PATHS = [
        '/kaggle/input/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset',
        '/kaggle/input/tuberculosis-tb-chest-xray-dataset',
        '/kaggle/input/tuberculosis-tb-chest-xray-dataset/TB_Chest_Radiography_Database',
    ]
    DATA = next((p for p in PREFERRED_DATA_PATHS if os.path.exists(p)), None)
    if DATA is None:
        for root, dirs, _ in os.walk('/kaggle/input'):
            if 'Normal' in dirs and 'Tuberculosis' in dirs:
                DATA = root; break

    REPO, MODEL_MODULE = None, None
    for cand in (glob.glob('/kaggle/input/datasets/*/*') + glob.glob('/kaggle/input/*')):
        p, mod = _find_model_module(cand)
        if p:
            REPO, MODEL_MODULE = p, mod
            break
    OUT = '/kaggle/working'
else:
    REPO, MODEL_MODULE = _find_model_module('.')
    if REPO is None:
        REPO, MODEL_MODULE = '.', 'tbnet_pytorch'  # fallback to error message below
    DATA = 'data/'
    OUT = '.'

print('REPO        :', REPO)
print('MODEL_MODULE:', MODEL_MODULE)
print('DATA        :', DATA)
print('OUT         :', OUT)

assert REPO and MODEL_MODULE, 'Cannot find model.py or tbnet_pytorch.py — attach tbnet-checkpoints dataset'
assert DATA, 'Cannot find the TB dataset — attach tuberculosis-tb-chest-xray-dataset'

sys.path.insert(0, REPO)
TBNet = __import__(MODEL_MODULE).TBNet

os.makedirs(os.path.join(OUT, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUT, 'models'), exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print('Device:', DEVICE)

In [ ]:
import cv2  # noqa: E402

# ── Preprocessing (inlined from src/preprocessing.py) ───────────────────────
# Mirrors preprocess_image() — must run on raw Kaggle PNGs to match training distribution.
def _is_padding(row):
    for i in range(6):
        if not np.count_nonzero(np.subtract(row, i)): return True
    for i in range(250, 256):
        if not np.count_nonzero(np.subtract(row, i)): return True
    return False

def preprocess_kaggle_image(path, image_size=(224, 224)):
    img_color = cv2.imread(path, 1)
    if img_color is None:
        raise FileNotFoundError(path)
    b, g, r = cv2.split(img_color)
    image = b.copy()
    h, w = image.shape
    min_y, max_y = 0, h - 1
    for row in image:
        if _is_padding(row): min_y += 1
        else: break
    for idx in range(h - 1, 0, -1):
        if _is_padding(image[idx]): max_y -= 1
        else: break
    image_T = image.T
    min_x, max_x = 0, w - 1
    for col in image_T:
        if _is_padding(col): min_x += 1
        else: break
    for idx in range(w - 1, 0, -1):
        if _is_padding(image_T[idx]): max_x -= 1
        else: break
    image = image[min_y:max_y, min_x:max_x]
    image = cv2.merge([image, image, image])
    image = cv2.resize(image, image_size, interpolation=cv2.INTER_LANCZOS4)
    mn, mx = float(np.min(b)), float(np.max(b))
    image = np.divide(np.subtract(image, mn), max(mx, 1e-6)) * 255.0
    return np.clip(image, 0, 255).astype(np.uint8)

# Cache preprocessed images so 5-epoch fine-tune doesn't repeat the slow Python loops.
_PREPROC_CACHE = {}
def get_preprocessed_pil(path):
    arr = _PREPROC_CACHE.get(path)
    if arr is None:
        arr = preprocess_kaggle_image(path)
        _PREPROC_CACHE[path] = arr
    return Image.fromarray(arr[:, :, 0], 'L')  # all 3 channels identical — keep grayscale L

# ── Filename index ──────────────────────────────────────────────────────────
def _build_index(data_root):
    idx = {}
    for dirpath, _, files in os.walk(data_root):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                idx[f] = os.path.join(dirpath, f)
                if f.startswith('Tuberculosis-'):
                    idx['TB-' + f[len('Tuberculosis-'):]] = os.path.join(dirpath, f)
                elif f.startswith('TB-'):
                    idx['Tuberculosis-' + f[len('TB-'):]] = os.path.join(dirpath, f)
    return idx
_IMG_INDEX = _build_index(DATA)
print(f'Indexed {len(_IMG_INDEX)} image keys under {DATA}')

# ── Resolve split CSV paths across new and legacy layouts ───────────────────
def _find_split_csv(name):
    candidates = [
        os.path.join(REPO, '..', 'data_splits', f'{name}.csv'),
        os.path.join(REPO, 'data_splits', f'{name}.csv'),
        os.path.join(REPO, f'{name}_split_new.csv'),
        os.path.join(REPO, f'{name}_split.csv'),
    ]
    for p in candidates:
        if os.path.exists(p):
            return os.path.abspath(p)
    raise FileNotFoundError(f'No split CSV found for "{name}"; tried {candidates}')

TRAIN_CSV = _find_split_csv('train')
VAL_CSV   = _find_split_csv('val')
TEST_CSV  = _find_split_csv('test')
print(f'train CSV: {TRAIN_CSV}')
print(f'val   CSV: {VAL_CSV}')
print(f'test  CSV: {TEST_CSV}')

# ── Dataset ─────────────────────────────────────────────────────────────────
TRANSFORM = transforms.Compose([
    transforms.ToTensor(),                  # (1, 224, 224) in [0,1] from grayscale PIL
    transforms.Normalize([0.5], [0.5]),
])

class TBDataset(Dataset):
    def __init__(self, csv_path, augment=None):
        self.df = pd.read_csv(csv_path, header=None, names=['filename', 'label'])
        self.augment = augment
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        name = os.path.basename(row['filename'])
        path = _IMG_INDEX.get(name)
        if path is None:
            alt = ('Tuberculosis-' + name[len('TB-'):]) if name.startswith('TB-') else \
                  ('TB-' + name[len('Tuberculosis-'):]) if name.startswith('Tuberculosis-') else None
            if alt: path = _IMG_INDEX.get(alt)
        if path is None:
            raise FileNotFoundError(f'Could not locate {name} under {DATA}')
        img = get_preprocessed_pil(path)
        if self.augment is not None:
            img = self.augment(img)
        return TRANSFORM(img), int(row['label'])

train_ds = TBDataset(TRAIN_CSV)
test_ds  = TBDataset(TEST_CSV)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0)
print(f'train={len(train_ds)}  test={len(test_ds)}')

_missing = 0
for i in range(min(20, len(test_ds))):
    try: _ = test_ds[i]
    except FileNotFoundError: _missing += 1
print(f'First-20 sanity: {20 - _missing}/20 resolved')

In [ ]:
# Eval helper used everywhere. mode='gray' → 1ch input (TBNet). mode='rgb' → repeat to 3ch (MobileNet student).
def evaluate(model, loader, label='', mode='gray'):
    model.eval()
    ys, ps, probs, lats = [], [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            if mode == 'rgb':
                imgs = imgs.repeat(1, 3, 1, 1)
            t0 = time.perf_counter()
            out = model(imgs)
            lats.append((time.perf_counter() - t0) * 1000 / imgs.size(0))
            p  = torch.softmax(out, 1)[:, 1].cpu().numpy()
            ps.extend(out.argmax(1).cpu().numpy())
            probs.extend(p)
            ys.extend(labels.numpy())
    cm = confusion_matrix(ys, ps)
    acc  = 100 * np.trace(cm) / cm.sum()
    sens = 100 * cm[1, 1] / cm[1].sum() if cm[1].sum() else 0
    spec = 100 * cm[0, 0] / cm[0].sum() if cm[0].sum() else 0
    auc  = roc_auc_score(ys, probs) if len(set(ys)) > 1 else float('nan')
    if label:
        print(f'  {label:30s}  Acc {acc:6.2f}  Sens {sens:6.2f}  Spec {spec:6.2f}  AUC {auc:.4f}  Lat {np.mean(lats):.2f} ms')
    return dict(acc=acc, sens=sens, spec=spec, auc=auc, lat=float(np.mean(lats)))

def load_tbnet(path):
    m = TBNet().to(DEVICE)
    sd = torch.load(path, map_location=DEVICE, weights_only=False)
    m.load_state_dict(sd, strict=False)
    return m

def load_student(path):
    m = mobilenet_v3_small(weights=None)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
    m = m.to(DEVICE)
    sd = torch.load(path, map_location=DEVICE, weights_only=False)
    m.load_state_dict(sd, strict=False)
    return m

# Resolve the models/ folder across both layouts: REPO/models/ (legacy) or REPO/../models/ (new).
MODELS_DIR = next((p for p in (
    os.path.join(REPO, 'models'),
    os.path.abspath(os.path.join(REPO, '..', 'models')),
) if os.path.isdir(p)), None)
print(f'MODELS_DIR: {MODELS_DIR}')

MODEL_FILES = {
    'TB-Net FP32':       (os.path.join(MODELS_DIR, 'tbnet_best.pth'),        load_tbnet,   'gray'),
    'TB-Net FP16':       (os.path.join(MODELS_DIR, 'tbnet_fp16.pth'),        load_tbnet,   'gray'),
    'TB-Net 25% pruned': (os.path.join(MODELS_DIR, 'tbnet_pruned_25.pth'),   load_tbnet,   'gray'),
    'TB-Net 50% pruned': (os.path.join(MODELS_DIR, 'tbnet_pruned_50.pth'),   load_tbnet,   'gray'),
    'MobileNet student': (os.path.join(MODELS_DIR, 'student_mobilenet.pth'), load_student, 'rgb'),
}
for n, (p, _, mode) in MODEL_FILES.items():
    print(f'{n:30s}  {"OK" if os.path.exists(p) else "MISSING"}  [{mode}]  {p}')

## Cell 2 — Sensitivity-gap investigation

Paper's claim: 99.86% acc / 100% sens / 99.71% spec. Ours: ~99.39 / 97.86 / 100.00.

Two things to check before writing it up:
1. **Is our split class-balanced the same way as Table 1?**
2. **Is the 2.14pp sensitivity gap stable, or seed/order noise?**

In [ ]:
# Split class-balance
for split_name, csv in [('train', TRAIN_CSV), ('val', VAL_CSV), ('test', TEST_CSV)]:
    df = pd.read_csv(csv, header=None, names=['f','label'])
    n0, n1 = (df.label == 0).sum(), (df.label == 1).sum()
    print(f'{split_name:5s}  Normal={n0:4d}  TB={n1:4d}  total={len(df):4d}  TB-frac={n1/len(df)*100:.1f}%')

print('\nBaseline (3 seeds, same weights):')
for seed in [0, 1, 2]:
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)
    m = load_tbnet(MODEL_FILES['TB-Net FP32'][0])
    evaluate(m, loader, f'seed={seed}')

**Interpretation hook for the paper:** if the three seeds give identical numbers, the gap is *not* eval noise — it's a retrain delta (different random init, augmentation, or LR schedule than the original TF1 training). State this honestly in the paper as "reproduction-from-scratch gap."

## Cell 3 — 75% pruning (adds 3rd sparsity point)

In [ ]:
def apply_pruning(model, amount):
    for mod in model.modules():
        if isinstance(mod, nn.Conv2d):
            prune.l1_unstructured(mod, name='weight', amount=amount)
    return model

def remove_pruning_masks(model):
    for mod in model.modules():
        if isinstance(mod, nn.Conv2d):
            try: prune.remove(mod, 'weight')
            except Exception: pass
    return model

def finetune(model, loader, epochs=5, lr=1e-4):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for e in range(epochs):
        tot, n = 0.0, 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(imgs), labels)
            loss.backward(); opt.step()
            tot += loss.item() * imgs.size(0); n += imgs.size(0)
        print(f'    epoch {e+1}/{epochs}  loss {tot/n:.4f}')

print('── 75% pruning + 5-epoch fine-tune ──')
m75 = load_tbnet(MODEL_FILES['TB-Net FP32'][0])
apply_pruning(m75, 0.75)
print('Before fine-tune:'); evaluate(m75, test_loader, 'pruned 75% (pre-FT)', mode='gray')
print('Fine-tuning...');     finetune(m75, train_loader, epochs=5)
remove_pruning_masks(m75)
p75_metrics = evaluate(m75, test_loader, 'pruned 75% (post-FT)', mode='gray')
p75_path = os.path.join(OUT, 'models', 'tbnet_pruned_75.pth')
torch.save(m75.state_dict(), p75_path)
MODEL_FILES['TB-Net 75% pruned'] = (p75_path, load_tbnet, 'gray')
print(f'Saved {p75_path}')

In [ ]:
# Regenerate sparsity_tradeoff figure with 4 points
print('── Re-evaluating all sparsity points ──')
sparsity_points = []
for name, sp in [('TB-Net FP32', 0), ('TB-Net 25% pruned', 25),
                 ('TB-Net 50% pruned', 50), ('TB-Net 75% pruned', 75)]:
    path, loader_fn, mode = MODEL_FILES[name]
    m = loader_fn(path)
    r = evaluate(m, test_loader, name, mode=mode)
    sparsity_points.append((sp, r))

xs   = [s for s, _ in sparsity_points]
accs = [r['acc']  for _, r in sparsity_points]
sens = [r['sens'] for _, r in sparsity_points]
spec = [r['spec'] for _, r in sparsity_points]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(xs, accs, 'o-', label='Accuracy')
ax.plot(xs, sens, 's--', label='Sensitivity')
ax.plot(xs, spec, '^:', label='Specificity')
ax.set_xlabel('Sparsity (%)'); ax.set_ylabel('Score (%)')
ax.set_title('Accuracy vs Sparsity (L1 pruning + 5-epoch fine-tune)')
ax.grid(True); ax.legend(); ax.set_ylim(80, 101)
plt.tight_layout()
fig_path = os.path.join(OUT, 'figures', 'sparsity_tradeoff.png')
plt.savefig(fig_path, dpi=150); plt.show()
print(f'Saved {fig_path}')

## Cell 4 — Phone-capture robustness

Apply CheXphoto-style augmentations to the test set and re-evaluate every model. This is the deployment-robustness story the abstract motivates: clinicians photographing X-ray films with phones.

Augmentations follow the CheXphoto natural-perturbation recipe: brightness shift, gaussian blur, additive moiré stripes, small rotation, glare patch.

In [ ]:
class PhoneCapture:
    """Deterministic phone-camera-style perturbation pipeline applied at PIL level."""
    def __init__(self, severity='moderate', seed=0):
        self.severity = severity
        self.rng = np.random.RandomState(seed)
    def __call__(self, img):
        img = img.convert('L')
        arr = np.array(img, dtype=np.float32)
        h, w = arr.shape
        # brightness shift
        arr = np.clip(arr * self.rng.uniform(0.7, 1.2) + self.rng.uniform(-15, 15), 0, 255)
        # additive moiré stripes
        freq = self.rng.uniform(4, 10)
        amp  = 10 if self.severity == 'moderate' else 18
        stripes = amp * np.sin(2 * np.pi * freq * np.arange(h)[:, None] / h)
        arr = np.clip(arr + stripes, 0, 255)
        # glare patch
        cx, cy = self.rng.randint(w//4, 3*w//4), self.rng.randint(h//4, 3*h//4)
        yy, xx = np.ogrid[:h, :w]
        r2 = (xx - cx) ** 2 + (yy - cy) ** 2
        glare = 60 * np.exp(-r2 / (2 * (min(h, w) / 6) ** 2))
        arr = np.clip(arr + glare, 0, 255)
        img = Image.fromarray(arr.astype(np.uint8))
        # gaussian blur
        img = img.filter(ImageFilter.GaussianBlur(radius=self.rng.uniform(0.6, 1.6)))
        # small rotation
        img = img.rotate(self.rng.uniform(-4, 4), resample=Image.BILINEAR, fillcolor=0)
        return img

phone_test_ds = TBDataset(TEST_CSV, augment=PhoneCapture(severity='moderate', seed=42))
phone_loader  = DataLoader(phone_test_ds, batch_size=32, shuffle=False, num_workers=2)

# Quick visual sanity check
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
clean_ds = TBDataset(TEST_CSV)
for j, idx in enumerate([0, 5, 10, 15]):
    c_img, lab = clean_ds[idx]
    p_img, _   = phone_test_ds[idx]
    axes[0, j].imshow(c_img.squeeze().numpy(), cmap='gray'); axes[0, j].set_title(f'clean (label={lab})'); axes[0, j].axis('off')
    axes[1, j].imshow(p_img.squeeze().numpy(), cmap='gray'); axes[1, j].set_title('phone-capture');         axes[1, j].axis('off')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'figures', 'phone_capture_examples.png'), dpi=120); plt.show()

In [ ]:
# Run every model on clean vs phone-capture; build the side-by-side table
rows = []
for name, (path, loader_fn, mode) in MODEL_FILES.items():
    if not os.path.exists(path):
        print(f'skip {name} (missing weights)'); continue
    m = loader_fn(path)
    print(f'\n── {name} ──')
    rc = evaluate(m, test_loader,  'clean', mode=mode)
    rp = evaluate(m, phone_loader, 'phone', mode=mode)
    rows.append({
        'model': name,
        'acc_clean': rc['acc'],  'acc_phone': rp['acc'],  'd_acc': rp['acc']  - rc['acc'],
        'sens_clean': rc['sens'],'sens_phone': rp['sens'],'d_sens': rp['sens']- rc['sens'],
        'spec_clean': rc['spec'],'spec_phone': rp['spec'],'d_spec': rp['spec']- rc['spec'],
        'auc_clean': rc['auc'],  'auc_phone': rp['auc'],
        'lat_ms': rc['lat'],
    })
df_results = pd.DataFrame(rows)
df_results.to_csv(os.path.join(OUT, 'results_phone.csv'), index=False)
df_results.round(2)

In [ ]:
# Visualize the robustness drop
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_results))
w = 0.35
ax.bar(x - w/2, df_results['sens_clean'], w, label='Sensitivity (clean)')
ax.bar(x + w/2, df_results['sens_phone'], w, label='Sensitivity (phone-capture)')
ax.set_xticks(x); ax.set_xticklabels(df_results['model'], rotation=20, ha='right')
ax.set_ylabel('Sensitivity (%)'); ax.set_ylim(0, 105)
ax.set_title('Phone-capture robustness — clean vs. perturbed test set')
ax.grid(axis='y', alpha=0.3); ax.legend()
plt.tight_layout()
out_fig = os.path.join(OUT, 'figures', 'phone_robustness.png')
plt.savefig(out_fig, dpi=150); plt.show()
print(f'Saved {out_fig}')

## Cell 5 — Final results table for the paper

In [ ]:
final = df_results[['model', 'acc_clean', 'sens_clean', 'spec_clean', 'auc_clean',
                    'acc_phone', 'sens_phone', 'spec_phone', 'auc_phone', 'lat_ms']].copy()
final.to_csv(os.path.join(OUT, 'results_clean.csv'), index=False)
print('Saved results_clean.csv + results_phone.csv')
final.round(2)

### What to put in the paper

1. **Reproduction delta** — class balance matches paper Table 1 (or doesn't — see Cell 2 output). Multi-seed eval is deterministic, so the sensitivity gap is a retrain delta, not noise.
2. **Sparsity curve** — `figures/sparsity_tradeoff.png` has 4 points (0/25/50/75). 75% number from Cell 3.
3. **Phone-capture robustness** — `results_phone.csv` + `figures/phone_robustness.png`. Story = which compressed model degrades least under phone-capture conditions.
4. **Deployment honesty** — keep ONNX/desktop-CPU latency numbers. Frame Android numbers as "projected, on-device benchmarking left as future work."
5. **INT8 note** — PyTorch INT8 .pth dropped from the table (quantized state-dict incompatibility). Compression story stands on FP16 + L1 pruning + MobileNetV3 distillation. ONNX INT8 (0.30 MB) is still the recommended deployment artifact and can be benchmarked separately via `onnxruntime`.
6. **Preprocessing** — every image is run through the paper's preprocessing pipeline (B-channel extraction, padding auto-crop, intensity rescale) before being fed to any model. Same pipeline the original TF1 codebase used. Inlined in the notebook for reproducibility.